In [12]:
import vlrdevapi as vlr
import pandas as pd
import re

events = vlr.event.list(tier='vct', return_all=True)

In [ ]:
# filter list of events down to only include international events, get their ID, and then add the actual event object to a list
intl_events = []
for event in events.events:
    if re.search(r"(Masters \w+|Champions 202\d|LOCK//IN)", event.name):
        intl_events.append(vlr.event(event.id))

intl_events

In [26]:
# create a list of all international matches
intl_matches = []
for event in intl_events:
    match_list = event.matches().matches
    for match in match_list:
        intl_matches.append(vlr.series(match.match_id))

intl_matches
TEST_intl_matches = [intl_matches[0]]

In [36]:
data_dict = {
    "columns": [
        "event_id", "event_name", "series_id", "map_id", "map_name",
        "team_name", "player_name", "player_id", "agent", 
        "acs", "kills", "deaths", "assists", "fkills", "fdeaths"
    ],
    "data": []
}

In [38]:
from vlrdevapi.exceptions import RateLimitError, RequestError

try:
    for seriesObj in intl_matches:
        series = seriesObj.info()
        eventID = series.event_id
        eventName = series.event_name
        seriesID = series.series_id
        for map in series.games:
            mapID = map.game_id
            stats = vlr.series.players(seriesID, mapID)
            mapName = stats.map_name
        for team in [stats.team1, stats.team2]:
            teamName = team.team_name
            for player in team.players:
                playerID = player.player_id
                playerName = player.name
                playerAgent = player.agents[0]
                playerACS = player.stats.overall.acs
                playerKills = player.stats.overall.kills
                playerDeaths = player.stats.overall.deaths
                playerAssists = player.stats.overall.assists
                playerFKs = player.stats.overall.first_kills
                playerFDs = player.stats.overall.first_deaths
                playerRow = [
                    eventID, eventName, seriesID, mapID, mapName,
                    teamName, playerName, playerID, playerAgent,
                    playerACS, playerKills, playerDeaths, playerAssists, playerFKs, playerFDs
                ]
                data_dict["data"].append(playerRow)
except RequestError:
    print("Error while requesting.")
except RateLimitError:
    print("Too many requests!")

IndexError: list index out of range